# Wavelet analysis for June experiment

In [18]:
import polars as pl
from pathlib import Path
import numpy as np
import pyboat
from pyboat import WAnalyzer
from pyboat import plotting
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from tqdm import tqdm
import pandas as pd
from scipy.signal import savgol_filter
import pandas as pd
import seaborn as sns

from dataclasses import dataclass
from typing import Generator

@dataclass
class WaveResults:
    """Wavlet analysis results"""
    ridge_df: pl.DataFrame
    summary_df: pl.DataFrame
    periods: np.ndarray
    fourier_df: pl.DataFrame

def add_period_marks(ax, x_min, x_max, interval=12):
    marks = np.arange(np.ceil(x_min / interval) * interval, x_max, interval)
    for mark in marks:
        ax.axvline(mark, color='gray', linewidth=0.5, linestyle=':', alpha=0.5)
    ax.set_xticks(marks)
    ax.set_xticklabels([f'{int(m)}h' for m in marks], fontsize=7)

def make_title(row: dict, well_col: str, meta_cols: list[str]) -> str:
    parts = [row[well_col]]
    for col in meta_cols:
        val = row.get(col)
        if val is not None and str(val).strip() != '' and str(val).lower() != 'nan':
            parts.append(str(val))
    return ' | '.join(parts)

def savgol_smooth(
    dataset: pl.DataFrame, 
    value_col: str,
    well_col: str,
    window_length: int = 13, 
    polyorder: int = 3
    ) -> pl.DataFrame:
    return (
        dataset
        .group_by(well_col)
        .map_groups(
            lambda g: 
                g.with_columns(
                    pl.Series(value_col, savgol_filter(
                        g[value_col].to_numpy(),
                        window_length=window_length,
                        polyorder=polyorder
                    ))
                )
        )
    )
    
def calculate_wavelets(
    dataset: pl.DataFrame,
    name_col: str, #this is like the group_label but the actual name of it
    well_list: list[str],
    well_col: str,
    stat_val: str,
    timepoint_col: str,
    sampling_interval: float = 1.0,
    period_range: tuple[int, int] = (18,40),
    n_periods: int=200,
    ridge_thresh: int=0,
) -> WaveResults:
    periods = np.linspace(period_range[0], period_range[1], n_periods)
    ridge_results = []
    fourier_results = []
    summary_rows = []
    
    for well in tqdm(well_list, desc='Wells'):
        well_df = (dataset
                   .filter(pl.col(well_col) == well)
                   .sort(timepoint_col)
                   )
        signal = well_df[stat_val].to_numpy()
        tpoints = well_df[timepoint_col].to_numpy()
        group = well_df[name_col].unique().item()
        
        # Wave analyzer
        wan = WAnalyzer(periods=periods, dt=sampling_interval, time_unit_label='h')
        
        # compute spectrum
        wan.compute_spectrum(signal, do_plot=False)
        
        ridge = wan.get_maxRidge(power_thresh=ridge_thresh)
        fourier_power = wan.get_averaged_spectrum()
        
        # ridge summaries
        ridge['time'] = tpoints
        ridge[well_col] = well
        ridge[name_col] = group
        ridge_results.extend(ridge.to_dict(orient='records'))
        
        for p, pw in zip(periods, fourier_power):
            fourier_results.append({
                well_col: well,
                name_col: group,
                'period': p,
                'power': pw
            })
        
        # summaries
        mid_idx = len(ridge) // 2
        summary_rows.append({
            well_col          : well,
            name_col          : group,
            'mean_period'     : ridge['periods'].mean(),
            'std_period'      : ridge['periods'].std(),
            'median_period'   : ridge['periods'].median(),
            'mean_amplitude'  : ridge['amplitude'].mean(),
            'std_amplitude'   : ridge['amplitude'].std(),
            'median_amplitude': ridge['amplitude'].median(),
            'mid_phase'       : ridge['phase'].iloc[mid_idx],
            'peak_fourier_period': periods[np.argmax(fourier_power)],
        })
    
    return WaveResults(
        ridge_df=pl.DataFrame(ridge_results),
        fourier_df=pl.DataFrame(fourier_results),
        periods=periods,
        summary_df=pl.DataFrame(summary_rows)
    )    
  
def plot_wavelets(
    wavelet_results: WaveResults,
    dataset: pl.DataFrame,
    well_list: list[str],
    well_col: str,
    name_col: str,
    stat_val: str,
    timepoint_col: str,
    win_len: int = 13,
    polyorder: int = 3,
    tick_intervals: int = 12
) -> Generator[plt.Figure]:
    
    ridge_df = wavelet_results.ridge_df.to_pandas()
    fourier_df = wavelet_results.fourier_df.to_pandas()
    periods = wavelet_results.periods
    
    for well in tqdm(well_list, desc='well plots'):
        well_signal = (
            dataset
            .filter(pl.col(well_col) == well)
            .sort(timepoint_col)
            .select([timepoint_col, stat_val])
            .to_pandas()
        )
        tpoints = well_signal[timepoint_col].to_numpy()
        signal = well_signal[stat_val].to_numpy()
        
        smoothed = savgol_filter(signal, window_length=win_len, polyorder=polyorder)
        well_signal[stat_val] = smoothed
        
        well_ridge = ridge_df[ridge_df[well_col] == well]
        well_fourier = fourier_df[fourier_df[well_col] == well]
        group = well_ridge[name_col].iloc[0]
        
        # plots
        fig, axs = plt.subplots(3, 1, figsize=(14,12))
        
        # detrended signal plot
        
        sns.lineplot(data=well_signal, 
                     x=timepoint_col, 
                     y=stat_val,
                     color='royalblue', 
                     linewidth=1.5,
                     ax=axs[0])
        axs[0].axhline(0, color='gray', linewidth=0.5, linestyle='--')
        axs[0].set_xlabel('Time (hours)')
        axs[0].set_ylabel('Expression')
        axs[0].set_title('Detrended Signal')
        axs[0].set_xlim(tpoints[0], tpoints[-1])
        add_period_marks(axs[0], tpoints[0], tpoints[-1], tick_intervals)
        
        # wavelet spectrum
        
        wan = WAnalyzer(periods=periods, 
                        dt=(tpoints[1] - tpoints[0]), 
                        time_unit_label='h')
        wan.compute_spectrum(signal, do_plot=False)
        extent = [tpoints[0], tpoints[-1], periods[-1], periods[0]]
        axs[1].imshow(np.abs(wan.transform), aspect='auto', extent=extent, 
                      cmap='viridis', origin='upper')
        axs[1].plot(well_ridge['time'], well_ridge['periods'],
                       color='cyan', linewidth=2, label='ridge')
        axs[1].set_xlabel('Time (hours)')
        axs[1].set_ylabel('Period (hours)')
        axs[1].set_title('Wavelet Power Spectrum')
        axs[1].legend(fontsize=8)
        axs[1].set_xlim(tpoints[0], tpoints[-1])  # ← force x limits explicitly
        axs[1].set_ylim(periods[-1], periods[0])  # ← force y limits explicitly
        add_period_marks(axs[1], tpoints[0], tpoints[-1], tick_intervals)
        
        # fourier spectrum
        peak_period = well_fourier.loc[well_fourier['power'].idxmax(), 'period']
        axs[2].plot(well_fourier['period'], well_fourier['power'],
                    color='darkred', linewidth=1.5)
        axs[2].axvline(peak_period, color='red', linewidth=1.5, linestyle='--',
                        label=f'Peak: {peak_period:.1f}h')
        axs[2].set_xlabel('Period (hours)')
        axs[2].set_ylabel('Power')
        axs[2].set_title('Fourier Spectrum (time-averaged)')
        axs[2].legend(fontsize=8)
        axs[2].set_xlim(periods[0], periods[-1])
        # add_period_marks(axs[2], periods[0], periods[-1])
        
        fig.suptitle(f"Well: {well} - {group}", fontsize=11)
        fig.tight_layout()
        
        yield fig

*Note*: The platemap processing was done in `06-01_envision_visualization.ipynb`

In [15]:
detrended_dataset = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live "
                        "Study/enVision/enVision_preprocessed/2026-06-01._.20"
                        "xREP._.PREPROCESSED.parquet")

plate_map = (
    pl.read_excel("/home/derfelt/PartchLabFiles/Circadian "
                  "Rhythms/Live Study/platemaps/2026_05_26"
                  "_JUNE-EXP_plate_map._.W-PERIODS.xlsx", engine='openpyxl')
)

timepoint_col = "Timepoint"
well_coll = "Well"
feature = "CalcResultI"
value = "mean"
meta_cols = ['group_label' , 'detected_period']
sampling_interval = 0.5

detrended_dataset = (
    pl.read_parquet(detrended_dataset)
    .filter(pl.col('feature') == "CalcResultI")
    .select(well_coll, timepoint_col, value)
    .join(plate_map, on=well_coll, how='left')
    .sort([timepoint_col, 'group_label'], descending=[False, False])
)

wells = detrended_dataset[well_coll].unique().to_list()

wave_res = calculate_wavelets(
    dataset=detrended_dataset,
    name_col='group_label',
    well_list=wells,
    well_col=well_coll,
    stat_val=value,
    timepoint_col=timepoint_col,
    sampling_interval=sampling_interval
)

Wells: 100%|██████████| 96/96 [00:01<00:00, 89.06it/s]


In [20]:
out_dir = Path("/home/derfelt/PartchLabFiles/Circadian "
               "Rhythms/Live Study/enVision/enVision_vi"
               "sual_results/2026-06-01")

out_file = out_dir / "06-01._.wavelet_analysis_visuals.pdf"

with PdfPages(out_file) as pdf:
    for fig in plot_wavelets(
        wavelet_results=wave_res,
        dataset=detrended_dataset,
        well_list=wells,
        well_col=well_coll,
        name_col='group_label',
        stat_val=value,
        timepoint_col=timepoint_col,
        tick_intervals=6
    ):
        pdf.savefig(fig)
        plt.close(fig)

well plots: 100%|██████████| 96/96 [00:21<00:00,  4.53it/s]
